# N133 · 计算几何与凸包

**目标：** 用向量关系替代不稳定斜率比较。

**先修：** N037, N126, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 点积/叉积；方向；共线；线段相交；面积；矩形/圆；凸包；整数精确判定。

## 从问题到算法

计算几何先把图形关系写成代数谓词。叉积符号判断左转、右转或共线；线段相交还需处理共线端点的包围盒。凸包通过不断删除不构成左转的中间点得到外边界。整数坐标用精确整数运算，避免不必要浮点斜率；共线计数用约分后的方向对。

## 最小实现

**本章接口：** `cross(o, a, b)`、`segments_intersect(a,b,c,d)`、`convex_hull(points)`、`max_points(points)`

In [1]:
from math import gcd
from collections import Counter

def cross(o,a,b): return (a[0]-o[0])*(b[1]-o[1])-(a[1]-o[1])*(b[0]-o[0])

def _on_segment(a,b,p):
    return cross(a,b,p)==0 and min(a[0],b[0])<=p[0]<=max(a[0],b[0]) and min(a[1],b[1])<=p[1]<=max(a[1],b[1])

def segments_intersect(a,b,c,d):
    x,y,z,w=cross(a,b,c),cross(a,b,d),cross(c,d,a),cross(c,d,b)
    if (x>0 and y<0 or x<0 and y>0) and (z>0 and w<0 or z<0 and w>0): return True
    return _on_segment(a,b,c) or _on_segment(a,b,d) or _on_segment(c,d,a) or _on_segment(c,d,b)

def rectangles_overlap(a,b):
    return max(a[0],b[0])<min(a[2],b[2]) and max(a[1],b[1])<min(a[3],b[3])

def convex_hull(points,include_collinear=False):
    points=sorted(set(map(tuple,points)))
    if len(points)<=2: return points
    if all(cross(points[0],points[-1],p)==0 for p in points): return points if include_collinear else [points[0],points[-1]]
    def half(sequence):
        stack=[]
        for p in sequence:
            while len(stack)>=2 and (cross(stack[-2],stack[-1],p)<0 if include_collinear else cross(stack[-2],stack[-1],p)<=0): stack.pop()
            stack.append(p)
        return stack
    return half(points)[:-1]+half(reversed(points))[:-1]

def max_points(points):
    best=0
    for i,(x,y) in enumerate(points):
        slopes=Counter(); duplicates=0
        for u,v in points[i+1:]:
            dx,dy=u-x,v-y
            if dx==dy==0: duplicates+=1; continue
            divisor=gcd(dx,dy); dx//=divisor; dy//=divisor
            if dx<0 or dx==0 and dy<0: dx,dy=-dx,-dy
            slopes[(dx,dy)]+=1
        best=max(best,1+duplicates+max(slopes.values(),default=0))
    return best

## 正确性、前提与复杂度

叉积等于有向平行四边形面积。凸包栈删除非左转点时，该点位于相邻两点与当前点包围的内侧，不能成为该半边的极点；上下两条单调链合成逆时针凸包。方向对用GCD约分并统一符号，竖直线和重复点均不需要浮点特例斜率。

**代价：** 凸包O(n log n)时间O(n)空间；最多共线点O(n²)次方向处理、O(n)辅助。默认凸包只保留极点，include_collinear=True返回所有边界输入点（适配围栏类题）。线段接触算相交；矩形只有正面积交叠才返回True。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

points=[(0,0),(1,0),(2,0),(2,2),(0,2),(1,1)]; hull=convex_hull(points)
show_table(['输入点','极点凸包成员','包含共线边界时成员'],[(p,p in hull,p in convex_hull(points,True)) for p in points])
show_table(['凸包有向边','内部测试点(1,1)叉积'],[((a,b),cross(a,b,(1,1))) for a,b in zip(hull,hull[1:]+hull[:1])])

输入点,极点凸包成员,包含共线边界时成员
"(0, 0)",True,True
"(1, 0)",False,True
"(2, 0)",True,True
"(2, 2)",True,True
"(0, 2)",True,True
"(1, 1)",False,False


凸包有向边,"内部测试点(1,1)叉积"
"((0, 0), (2, 0))",2
"((2, 0), (2, 2))",2
"((2, 2), (0, 2))",2
"((0, 2), (0, 0))",2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert segments_intersect((0,0),(2,2),(0,2),(2,0))
assert segments_intersect((0,0),(1,0),(1,0),(2,0))
assert not segments_intersect((0,0),(1,0),(2,0),(3,0))
assert not rectangles_overlap([0,0,1,1],[1,0,2,1])
assert max_points([(1,1),(1,1),(2,2)])==3
assert convex_hull([(0,0),(1,1),(2,2)])==[(0,0),(2,2)]
assert len(convex_hull([(0,0),(1,1),(2,2)],True))==3
from random import Random
rng=Random(133)
for _ in range(250):
    points=[(rng.randrange(-3,4),rng.randrange(-3,4)) for _ in range(rng.randrange(1,10))]; hull=convex_hull(points)
    if len(hull)>=3:
        assert all(cross(a,b,p)>=0 for a,b in zip(hull,hull[1:]+hull[:1]) for p in points)
        assert all(cross(hull[i-2],hull[i-1],hull[i])>0 for i in range(len(hull)))
    elif len(hull)==2: assert all(_on_segment(hull[0],hull[1],p) for p in points)
    else: assert len(set(points))==1
    want=max(Counter(points).values())
    for i,a in enumerate(points):
        for b in points[i+1:]:
            if a!=b: want=max(want,sum(cross(a,b,p)==0 for p in points))
    assert max_points(points)==want
print('N133: 所有本章断言通过')

N133: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 保留所有边界共线点。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用规范化gcd方向替代浮点斜率。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 149. Max Points on a Line（canonical）
- 587. Erect the Fence（canonical）
- 836. Rectangle Overlap（canonical）
- 223. Rectangle Area（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。